# Case Study 2: The G10 FX Carry Trade

**Name:**
**Data as-of date:** *(TODO: fill in after your final run; FRED/Yahoo update over time)*

Skeleton notebook following the assignment's Sections 0–6. Everything marked `TODO` is yours to fill in.
Cells marked **Checkpoint** are sanity checks: run them and make sure the output looks reasonable before moving on.

| Section | Points |
|---|---|
| 1. Data pipeline | 15 |
| 2. Modules 3–4, basket-wide | 15 |
| 3. Modules 5–6, basket-wide | 20 |
| 4. Basket construction | 25 |
| 5. Backtest and tearsheet | 25 |
| 6. Bonus: trend overlay | +5 |

## Rookie-Mistake Checklist

Tick these off as you go. Each one appears again in the section where it's most likely to bite.

- [ ] **Trade direction / sign.** One quoting convention for every spot series (Section 1.2).
- [ ] **Accrual convention.** One convention, documented, never switched (Section 1.4).
- [ ] **Risk-adjusted sizing**, not raw-yield ranking alone (Section 4.2).
- [ ] **No look-ahead in vol forecasts.** The weight held on day $t+1$ uses information through day $t$ (Sections 4.2, 5.1).
- [ ] **Funding spread** included (Section 4.3).
- [ ] **Correlation of legs** checked; ten currencies is not ten independent bets (Section 4.4).
- [ ] **Skewness, kurtosis, and drawdown** reported, not just Sharpe (Section 5.2).
- [ ] **Reproducible cache** with a documented as-of date (Section 1.3).

---
# 0. Before You Start: What's Already Built

What the course's `fetch_data.py` does for FX (the parts you're extending):

| Step | What it does | Carry it forward? |
|---|---|---|
| Spot | `yf.download` of 6 tickers (`AUDUSD=X`, `USDJPY=X`, `AUDJPY=X`, `EURUSD=X`, `GBPUSD=X`, `NZDUSD=X`) from 2010-01-01, `["Close"]` | Yes, extend to 9 pairs vs USD |
| Rates | FRED via the CSV endpoint `fredgraph.csv?id=...` (no API key, no extra library), `na_values="."` | Yes, same endpoint for all 10 |
| Alignment | Monthly rates `reindex(..., method="ffill")` onto daily FX dates | Yes, but see the timing note in 1.4 |
| Accrual | `accrued_pct`: **compounded** over **actual calendar days**, $\left[(1+i/100)^{d/365}-1\right]\times 100$ | Yes, same convention (handout: don't switch) |
| Carry P&L | spot log return (%) + accrued(AUD) − accrued(JPY) | Generalize to accrued(foreign) − accrued(USD) |
| Caching | None: it re-downloads every run and overwrites the CSVs | You add this (1.3) |

**Things in `fetch_data.py` to check before reusing (each one is a rookie-mistake candidate):**

1. **Column labels.** It does `fx.columns = list(fx_tickers.keys())`, which assumes `yf.download` returns columns in the same order you passed them. Recent `yfinance` versions return them **sorted by ticker** instead, which would silently mislabel every series. Check by printing `fx.tail()`: AUD/JPY should be around 90–110, USD/JPY around 140–160, EUR/USD around 1.0–1.2. In your version, rename by **ticker**, not by position.
2. **Quote direction is mixed.** `USDJPY` is JPY per USD; the others are USD per foreign. You'll need one convention (1.2).
3. **Rate timing.** FRED dates each monthly average at the 1st of the month, so forward-filling puts March's *average* (published in April) onto March 1. That's a small look-ahead (1.4).
4. **Which day's rate accrues.** The rate reindexed onto date $t$ is used for the interval $(t-1, t]$. That only matters on month boundaries, but think about which is right.
5. **Sample start 2010.** That excludes the 2008 unwind the handout mentions. Yahoo FX history goes back further for most pairs; decide whether to start earlier (1.2).

**TODO (prose):** In your own words, 3–5 bullets on what you're keeping from `fetch_data.py` and what you're changing.

In [1]:
import warnings, importlib
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import yfinance as yf

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
import statsmodels.api as sm
from arch import arch_model

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (10, 4)

RNG = np.random.default_rng(5005)           # one seed for the whole notebook (bootstrap, etc.)
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
START = "2010-01-01"                        # fetch_data.py's choice; TODO: earlier if you want 2008 in sample

CCYS     = ["EUR", "JPY", "GBP", "CHF", "CAD", "AUD", "NZD", "SEK", "NOK"]   # nine non-USD currencies
ALL_CCYS = ["USD"] + CCYS

---
# 1. Data Pipeline: Building the G10 Universe (15 pts)

## 1.1 Short-Term Interest Rates

Same source as `fetch_data.py`: `https://fred.stlouisfed.org/graph/fredgraph.csv?id=<SERIES_ID>`.

**Verify every ID on fred.stlouisfed.org first.** For each one, check that it exists, when its last observation is, its frequency, and its units (% p.a.).

| Currency | FRED series | Frequency | Last obs | Notes / substitutions |
|---|---|---|---|---|
| USD | | | | |
| EUR | | | | Euro Area aggregate |
| JPY | `IRSTCI01JPM156N` | monthly | | from `fetch_data.py` |
| GBP | | | | |
| CHF | | | | |
| CAD | | | | may need the `IR3TIB01...` family |
| AUD | `IRSTCI01AUM156N` | monthly | | from `fetch_data.py` |
| NZD | | | | |
| SEK | | | | |
| NOK | | | | |

**TODO (prose):** Justify any substitution. If you mix overnight call-money and 3-month interbank rates, say what bias that introduces (term premium) and whether it matters for *ranking*.

In [20]:
FRED_IDS = {                     # column name -> FRED series id (same naming style as fetch_data.py)
    "USD_rate": "IRSTCI01USM156N",            # United states
    "EUR_rate": "IRSTCI01EZM156N",            # EU 
    "JPY_rate": "IRSTCI01JPM156N",
    "GBP_rate": "IRSTCI01GBM156N",            # Great Britain
    "CHF_rate": "IRSTCI01CHM156N",            # Switzerland
    "CAD_rate": "IRSTCI01CAM156N",            # Australia
    "AUD_rate": "IRSTCI01AUM156N",
    "NZD_rate": "IRSTCI01NZM156N",            # New Zeland
    "SEK_rate": "IRSTCI01SEM156N",            # Sweden
    "NOK_rate": "IRSTCI01NOM156N",            # Norway
}

def fred_series(series_id):
    # Same pattern as fetch_data.py
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    return pd.read_csv(url, index_col=0, parse_dates=True, na_values=".").iloc[:, 0]

When extracting the “Immediate Rates (<24 Hours): Call Money/Interbank Rate” data from FRED I was able to find a series associated with each of the specified G10 currencies. I will keep an eye out for any issues with these data series, but for the moment they all seem to be available and consistent. 

## 1.2 Spot FX Rates

**One convention everywhere:** $S_{i,t}$ = **USD per 1 unit of currency $i$** (the `XXXUSD` direction, like `AUDUSD` in `fetch_data.py`). Then $\Delta s_{i,t} > 0$ means currency $i$ appreciated, which is a gain for **long $i$ / short USD**.

Pairs quoted the other way (`USDJPY=X` = JPY per USD) get inverted: $S = 1/\text{quote}$, i.e. $\log S = -\log(\text{quote})$.

| Currency | Yahoo ticker | Quote is | Invert? | Sanity value (approx.) |
|---|---|---|---|---|
| EUR | `EURUSD=X` | USD per EUR | no | ~1.1 |
| JPY | `USDJPY=X` | JPY per USD | **yes** | ~0.007 after inverting |
| GBP | `GBPUSD=X` | | | |
| CHF | | | | |
| CAD | | | | |
| AUD | `AUDUSD=X` | USD per AUD | no | ~0.65 |
| NZD | `NZDUSD=X` | | | |
| SEK | | | | |
| NOK | | | | |

Also keep `AUDJPY=X` (from `fetch_data.py`) for the single-pair comparison in 2.3 and 4.4.

In [23]:
YF_TICKERS = {
    # ccy: (ticker, invert)
    "EUR": ("EURUSD=X", False),
    "JPY": ("JPYUSD=X", False),
    "AUD": ("AUDUSD=X", False),
    "GBP": ("GBPUSD=X", False),
    "CHF": ("CHFUSD=X", False),
    "CAD": ("CADUSD=X", False),
    "NZD": ("NZDUSD=X", False),
    "SEK": ("SEKUSD=X", False),
    "NOK": ("NOKUSD=X", False)
}

I decided to keep everything in USD basis, so an upward move in any of these tickers always means the foreign currency strengthened. This will be my standard approach throughout the case study. 


## 1.3 The Cache

`fetch_data.py` re-downloads everything every run. Your version should:
- read from `data/*.csv` if present **and** fresh enough,
- otherwise fetch, save, and say so,
- print a one-line summary (rows, first date, last date, cached vs. fetched).

Daily Yahoo data and monthly FRED data need **different** staleness rules: a monthly series whose last row is 6 weeks old may still be fully up to date.

**TODO (prose):** Your invalidation rule, and what happens on weekends/holidays. Add a `force=True` escape hatch for a deliberate refresh.

In [24]:
def load_or_fetch(name, fetch_fn, max_staleness_days, force=False):
    '''
    name               : file stem -> data/{name}.csv
    fetch_fn           : zero-arg function returning a DataFrame with a DatetimeIndex
    max_staleness_days : refetch if the last row is older than this many days
    '''
    path = DATA_DIR / f"{name}.csv"
    if path.exists() and not force:
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        if (pd.Timestamp.today() - df.index.max()).days <= max_staleness_days:
            print(f"Loaded {name} from {path} (last row: {df.index.max().date()})")
            return df
        else:
            print(f"{name} is stale (last row: {df.index.max().date()}), refetching...")
            df = fetch_fn()
            df.to_csv(path)
            print(f"Saved {name} to {path} (last row: {df.index.max().date()})")
            return df
    else:
        print(f"{name} not found or force=True, fetching...")
    df = fetch_fn()
    df.to_csv(path)
    print(f"Saved {name} to {path} (last row: {df.index.max().date()})")
    return df
    #   1. if path exists and not force: read it (index_col=0, parse_dates=True)
    #   2. stale? compare df.index.max() to pd.Timestamp.today()
    #   3. if stale or missing: df = fetch_fn(); df.to_csv(path)
    #   4. print summary; return df
    ...

def fetch_rates():
    """Download every series in FRED_IDS; return one DataFrame (monthly, % p.a.)."""
    # 1. Fail early and clearly if any ID is still a placeholder
    missing = [col for col, sid in FRED_IDS.items() if sid is None]
    if missing:
        raise ValueError(f"Fill in FRED_IDS first. Still missing: {missing}")

    # 2. Download each series, and say which one broke if a download fails
    series = {}
    for col, sid in FRED_IDS.items():
        try:
            s = fred_series(sid)
        except Exception as e:
            raise RuntimeError(f"Download failed for {col} ({sid}): {e}") from e
        print(f"{col:<9} {sid:<18} {s.first_valid_index():%Y-%m} to "
              f"{s.last_valid_index():%Y-%m}  n={s.notna().sum()}")
        series[col] = s

    # 3. Combine side by side on the union of all dates
    return pd.concat(series, axis=1, sort=True)    
    

def fetch_spot():
    """Daily spot closes for the nine G10 currencies, in USD per unit of foreign currency."""
    tickers = [t for t, _ in YF_TICKERS.values()]
    raw = yf.download(tickers, start=START, auto_adjust=True, progress=False)["Close"]

    # Fail loudly if Yahoo returned nothing for a ticker
    bad = [t for t in tickers if t not in raw.columns or raw[t].isna().all()]
    if bad:
        raise RuntimeError(f"No data returned for: {bad}")

    # Build the output column by column, selecting each ticker BY NAME
    spot = pd.DataFrame(index=raw.index)
    for ccy, (ticker, invert) in YF_TICKERS.items():
        spot[ccy] = 1 / raw[ticker] if invert else raw[ticker]

    spot.index.name = "Date"
    return spot
    

In [25]:
rates = load_or_fetch("g10_rates", fetch_rates, max_staleness_days=30)
spot  = load_or_fetch("g10_spot",  fetch_spot,  max_staleness_days=30)

# Checkpoint:
print(rates.tail())       # all in % p.a.? any currency stuck at an old date?
print(spot.tail())        # compare against the sanity column in the 1.2 table
print(spot.isna().sum())  # holidays differ across countries

g10_rates is stale (last row: 2026-08-01), refetching...
USD_rate  IRSTCI01USM156N    1954-07 to 2026-08  n=866
EUR_rate  IRSTCI01EZM156N    1994-01 to 2026-01  n=385
JPY_rate  IRSTCI01JPM156N    1985-07 to 2026-08  n=494
GBP_rate  IRSTCI01GBM156N    1978-01 to 2026-08  n=584
CHF_rate  IRSTCI01CHM156N    1972-01 to 2024-03  n=627
CAD_rate  IRSTCI01CAM156N    1975-01 to 2026-08  n=620
AUD_rate  IRSTCI01AUM156N    1990-08 to 2026-08  n=433
NZD_rate  IRSTCI01NZM156N    1985-01 to 2024-12  n=480
SEK_rate  IRSTCI01SEM156N    1955-01 to 2020-10  n=790
NOK_rate  IRSTCI01NOM156N    1982-01 to 2026-08  n=536
Saved g10_rates to data/g10_rates.csv (last row: 2026-08-01)
g10_spot not found or force=True, fetching...
Saved g10_spot to data/g10_spot.csv (last row: 2026-10-14)
                  USD_rate  EUR_rate  JPY_rate  GBP_rate  CHF_rate  CAD_rate  \
observation_date                                                               
2026-04-01        3.640000       NaN     0.727    3.7300       NaN 

## 1.4 Constructing Per-Currency Carry P&L

Generalize the last block of `fetch_data.py`. For **long currency $i$, funded in USD**, over the interval ending on day $t$:

$$
z_{i,t} = \underbrace{100\,\Delta \log S_{i,t}}_{\text{spot log return (\%)}} + \text{accrued}(i_i, d_t) - \text{accrued}(i_{\text{USD}}, d_t),
\qquad
\text{accrued}(i, d) = \left[\left(1 + \tfrac{i}{100}\right)^{d/365} - 1\right] \times 100
$$

with $d_t$ = calendar days since the previous row (Monday has $d_t = 3$). This is exactly `fetch_data.py`'s convention, with USD replacing JPY as the funding leg. **Keep it** (compounded, actual/365) and state it once.

Decisions to document:
1. **Rate timing.** `fetch_data.py` forward-fills the monthly average onto the same month's days. Options: (a) keep it, for comparability with the lecture; (b) lag one month (`rates.shift(1)` at monthly frequency before forward-filling) to remove the look-ahead. For the backtest in Sections 4–5, (b) is the defensible choice.
2. **Which day's rate accrues** over $(t-1, t]$: the value aligned to $t-1$ (when you held the position) or to $t$ (what `fetch_data.py` does)?
3. **NaN handling.** Drop rows where any currency is missing, or compute each currency's series on its own valid dates? Forward-filling spot creates a fake 0% day followed by a two-day return.

In [ ]:
def accrued_pct(annual_rate_pct, n_days, day_count=365):
    # Same as fetch_data.py: compounded accrual over actual calendar days, in %
    return ((1 + annual_rate_pct / 100) ** (n_days / day_count) - 1) * 100

# TODO: spot log returns in % (DataFrame, one column per currency in CCYS)
spot_ret = ...

# TODO: monthly -> daily rates on spot_ret's index (decide: lag one month or not)
rates_daily = ...

# TODO: calendar days between consecutive rows
calendar_days = ...      # e.g. spot_ret.index.to_series().diff().dt.days

# TODO: carry DataFrame: for each ccy,
#       spot_ret[ccy] + accrued_pct(rates_daily[f"{ccy}_rate"], calendar_days)
#                     - accrued_pct(rates_daily["USD_rate"],   calendar_days)
carry = ...

# Also needed later: the annualized differential vs USD known on each date (for ranking and the Fama regression)
diff_vs_usd = ...        # DataFrame, % p.a., columns = CCYS

In [ ]:
# Checkpoint:
# 1. carry.shape, carry.index.min(), carry.index.max(), carry.isna().sum()
# 2. Sign: on a day EUR clearly strengthened vs USD, carry["EUR"] > 0. On a day the yen strengthened
#    (USDJPY quote fell), carry["JPY"] > 0. If the JPY sign is backwards, the inversion didn't happen.
# 3. Accrual: (carry - spot_ret).sum() / (calendar_days.sum() / 365) should be close to diff_vs_usd.mean() (% p.a.)
# 4. Continuity with the lecture: carry["AUD"] - carry["JPY"] should track data/carry_pnl.csv closely
#    (not exactly: the AUDJPY=X cross and the two USD legs are separate quotes, and your rate timing may differ). Why is the difference of two USD-funded legs ≈ AUD/JPY carry?

---
# 2. Revisiting Modules 3–4: Is the Single-Pair Story Robust? (15 pts)

## 2.1 Random Walk / Stationarity Check, Basket-Wide

Two complementary pieces of evidence for each of the nine spot series (the USD column is the numeraire):
- **Visual:** SACF of log price vs. SACF of log returns (Module 4, Part 5).
- **Formal:** an ADF test (`adfuller`) on log price and on log returns. The null is a unit root (random walk), so a small p-value means stationary.

**Think about:** CHF had a floor against EUR (2011–2015) and DKK-style pegs exist in the G10 neighborhood. What would a managed or pegged currency look like in these diagnostics?

In [ ]:
# TODO: grid of SACF plots, e.g. 9 rows x 2 cols (log price | log return), or two 3x3 grids

In [ ]:
# TODO: summary table: currency | ADF stat (log price) | p-value | ADF stat (log return) | p-value
adf_table = ...

**Interpretation (TODO):** Do all log prices look like random walks? Any exceptions, and when?

## 2.2 Mean Carry Return: Basket-Wide Significance

Module 4 IID test for each currency: $\bar z$, $\text{SE} = s/\sqrt{n}$, $t = \bar z / \text{SE}$.

Also report the **average rate differential** per currency, so you can check the forward-premium-puzzle question: do the highest-differential currencies have the most robust positive carry?

In [ ]:
# TODO: table: ccy | mean daily carry (%) | SE | t-stat | annualized mean (%) | avg differential vs USD (% p.a.)
mean_table = ...

In [ ]:
# TODO (recommended): scatter of avg differential (x) vs annualized mean carry (y), labeled by currency

**Interpretation (TODO):** Which currencies are significant? Does carry P&L line up with the differential? Connect this to UIP and the forward premium puzzle.

## 2.3 Bootstrap Robustness

Pick 2–3 currencies, including AUD/JPY for continuity. Resample **days with replacement** 500–1000 times and record the mean each time.

**Think about:** the IID bootstrap assumes days are exchangeable. Given volatility clustering (Module 6), is that assumption fair? (A block bootstrap is a natural extension, worth a sentence even if you don't implement it.)

In [ ]:
def bootstrap_mean(x, B=1000, rng=RNG):
    # TODO: return an array of B bootstrap sample means
    ...

# TODO: AUD/JPY series for continuity: data/carry_pnl.csv from fetch_data.py, rebuilt from the AUDJPY column,
#       or carry["AUD"] - carry["JPY"]. Say which.
# TODO: histograms of bootstrap means, with a vertical line at 0 and at the sample mean
# TODO: report the share of bootstrap means <= 0 and a 95% percentile interval

**Interpretation (TODO):** Is the AUD/JPY finding robust or fragile? How does it compare to the other currencies you picked?

---
# 3. Revisiting Modules 5–6: Mean Predictability and Volatility per Currency (20 pts)

## 3.1 ARMA Identification, Basket-Wide
For each currency: ACF/PACF, then an AIC/BIC grid over $p, q \in \{0,\dots,3\}$ (Module 5).

In [ ]:
def arma_grid(x, max_p=3, max_q=3):
    # TODO: return a DataFrame of (p, q, AIC, BIC); use try/except around each fit
    ...

# TODO: loop over currencies, keep the best order by BIC (and by AIC), and collect a summary table:
#       ccy | best (p,q) by BIC | best (p,q) by AIC | any AR/MA coefficient significant?
arma_summary = ...

In [ ]:
# TODO: compact ACF/PACF grid for all nine currencies

**Interpretation (TODO):** Does "no meaningful mean predictability" hold basket-wide? Any outliers? Is a significant coefficient also *economically* meaningful?

## 3.2 GARCH Fitting, Basket-Wide

For each currency, fit GARCH(1,1) with Normal and with Student-$t$ innovations.

**Reminders from Module 6:**
- `res.conditional_volatility` is already a standard deviation.
- Implied unconditional variance is $\hat\alpha_0 / (1 - \hat\alpha_1 - \hat\beta_1)$.
- These are **full-sample** fits: fine for description here, but they contain look-ahead bias if used to trade (that's what Section 5.1 fixes).

In [ ]:
garch_fits = {}   # ccy -> {"normal": res, "t": res}

# TODO: loop over currencies, fit both, store them

In [ ]:
# TODO: summary table:
#   ccy | alpha1 | beta1 | persistence | nu (t fit) | AIC normal | AIC t | BIC normal | BIC t | winner
#       | Ljung-Box p-value on std_resid**2 (e.g. lag 10)
garch_summary = ...

**Interpretation (TODO):** Do all nine show volatility clustering? Which are most persistent? Does the $t$ distribution win everywhere? Any fits with leftover dependence?

## 3.3 The Fama Regression, Basket-Wide

$$
\Delta s_{t+1} = \alpha + \beta\,(i_{\text{foreign},t} - i_{\text{USD},t}) + \varepsilon_{t+1}
$$

**Before coding, derive the UIP prediction for $\beta$ under *your* quoting convention.** UIP says the expected return on the carry position is zero. With $S$ = USD per unit of foreign currency, work out $E_t[\Delta s_{t+1}]$ in terms of $i_{\text{foreign}} - i_{\text{USD}}$ and check whether UIP implies $\beta = +1$ or $\beta = -1$ in this regression. If it doesn't match the handout, either flip the regressor or the spot convention, and say so explicitly. This is exactly the sign-error trap.

**Practical choices to document:**
- **Units must match.** Either use daily $\Delta s$ with the differential converted to a daily rate, or (often cleaner, since the rates are monthly) sample both at **monthly** frequency with the annual differential divided by 12.
- **Standard errors.** Residuals are heteroskedastic (GARCH!). Consider HAC (Newey–West) SEs: `.fit(cov_type="HAC", cov_kwds={"maxlags": ...})`.
- The interesting test is $H_0: \beta = 1$ (UIP), not just $\beta = 0$.

In [ ]:
# TODO: for each currency, build y (spot change) and x (lagged differential) at your chosen frequency,
#       run OLS with HAC SEs, and collect:
#       ccy | beta_hat | SE | t-stat for beta=0 | t-stat for beta=1 (UIP) | n
fama_table = ...

**Interpretation (TODO):** Does the forward premium puzzle show up basket-wide? Explain why this table justifies expecting positive P&L from a cross-sectional carry basket.

---
# 4. Building the G10 Carry Basket (25 pts)

**Shared machinery.** Every basket in Sections 4–6 can be written as a daily weight matrix $W$ (dates × 9 currencies), with daily P&L

$$
r^{\text{basket}}_{t} = \sum_i w_{i,t-1}\, z_{i,t}.
$$

A negative weight is a short position in currency $i$ vs. USD. It loses the differential and gains if $i$ depreciates, which the sign handles automatically.
Building $W$ once and reusing one P&L/metrics function keeps all strategies comparable.

**Simplification to note:** holding *weights* fixed for a month implicitly assumes daily rebalancing to those weights; holding *notionals* fixed lets weights drift. Either is acceptable if you document it.

In [ ]:
def month_start_rebalance_dates(index):
    # TODO: first trading day of each month in `index`
    ...

def basket_pnl(W, carry):
    '''W: weights decided at each date's close, applied to the NEXT day's return.'''
    # TODO: (W.shift(1) * carry).sum(axis=1), with NaN handling
    ...

def perf_stats(r, freq=252):
    # TODO: dict with ann. return, ann. vol, Sharpe, skew, excess kurtosis, max drawdown
    ...

## 4.1 Rank-and-Sort Basket
At each month start: rank the nine currencies by the differential vs USD **known at that date**. Long the top 3 (+1/6 each), short the bottom 3 (−1/6 each), so the long notional equals the short notional (+0.5 / −0.5 gross per side; choose and document your scaling).

In [ ]:
# TODO: build W_rank (dates x ccys): set weights on rebalance dates, then forward-fill until the next one
W_rank = ...
pnl_rank = ...

# Checkpoint: W_rank.sum(axis=1) should be ~0 (dollar-neutral); (W_rank != 0).sum(axis=1) should be 6

In [ ]:
# TODO: heatmap or table of which currencies were long/short over time (shows the basket's composition drift)

## 4.2 Volatility-Targeted (Risk-Parity) Basket

Within each leg, weight $\propto 1/\hat\sigma_{i}$, then rescale so each leg still sums to ±0.5 (or to a target volatility; document the choice).

**Timing:** the weight held over day $t+1$ must use $\hat\sigma_{i,t+1|t}$, which is built from returns **through day $t$** only.
- With `arch`, `res.conditional_volatility[t]` is $\sigma_{t|t-1}$: the volatility *for* day $t$, built from data through $t-1$.
- Write out which index of `conditional_volatility` (or `res.forecast(...)`) corresponds to "known at the close of day $t$, used for day $t+1$," then check that `basket_pnl`'s `shift(1)` isn't shifting a second time.

*(Using the full-sample fits from 3.2 here still leaks parameter information. Note it; Section 5.1 removes it.)*

In [ ]:
# TODO: sigma_fcst DataFrame (dates x ccys) of one-step-ahead vol forecasts, aligned so row t = known at close of t
sigma_fcst = ...

# TODO: build W_vol from W_rank's long/short membership, using inverse-vol weights within each leg
W_vol = ...
pnl_vol = ...

# Checkpoint: leg sums still +0.5 / -0.5; no NaNs in the first rows after the warm-up

## 4.3 Adding Realism: Costs and Funding Spread

- **Transaction cost:** at each rebalance, turnover $= \sum_i |w_{i,\text{new}} - w_{i,\text{old}}|$ and cost $= \text{bps}/10^4 \times \text{turnover}$, charged on the rebalance day.
- **Funding spread:** each long leg earns (differential − spread); each short leg pays (differential + spread). Either way you lose $\text{spread} \times d_t/365$ per unit of **gross** exposure per day.

Run a small grid, e.g. costs ∈ {0, 5, 10} bps and spread ∈ {0, 10, 25} bps p.a.

In [ ]:
def apply_frictions(W, carry, d, cost_bps, spread_bps_pa):
    # TODO: return net daily P&L = gross P&L - transaction costs - funding drag
    ...

# TODO: grid table of net Sharpe / ann. return for both baskets across (cost_bps, spread_bps_pa)
friction_grid = ...

**Interpretation (TODO):** How much gross carry survives realistic frictions? Which basket is more sensitive, and why (turnover)?

## 4.4 Basket vs. Single Pair
Same sample period for all three: (a) AUD/JPY single pair, (b) equal-weight rank-and-sort, (c) vol-targeted.

In [ ]:
# TODO: perf_stats table for the three strategies (common date range!)
# TODO: cumulative P&L plot of the three

In [ ]:
# TODO: correlation matrix (heatmap) of the nine carry legs, and optionally a rolling average pairwise correlation
#       (addresses the "ten currencies != diversified" rookie mistake)

**Interpretation (TODO):** Does the basket improve risk-adjusted returns vs. the single pair, and by how much? What does the correlation structure say about real diversification?

---
# 5. Full Backtest and Risk Tearsheet (25 pts)

## 5.1 Walk-Forward Loop

At each monthly rebalance date $t$, using **only data through $t$**:
1. rank currencies by the differential known at $t$,
2. refit GARCH on an expanding window **if** a refit is due (choose a frequency: monthly, quarterly, or annually, and justify it),
3. between refits, update $\hat\sigma$ daily with the variance recursion and the *fixed* most recent parameters,
4. compute the vol-targeted weights, trade into them, and charge costs and funding spread.

**Choices to document:** initial training window (e.g. 2–3 years), refit frequency, and the `arch` approach. Useful tools: `arch_model(...).fit(last_obs=date)` fits only on data before `last_obs`; `res.forecast(start=...)` produces forecasts beyond it with those parameters.

**Checkpoint idea:** add an `assert` that every fit's training data ends on or before its rebalance date.

In [ ]:
REFIT_EVERY = ...      # TODO, e.g. "QS" (quarterly) and justify
WARMUP_YEARS = ...     # TODO
COST_BPS, SPREAD_BPS_PA = ..., ...   # TODO: your base-case frictions

def walk_forward(carry, rates_daily, d):
    '''
    Returns: W_wf (dates x ccys), net daily P&L, turnover series, costs series, sigma forecasts used
    '''
    # TODO: loop over rebalance dates
    #   - skip until warm-up is complete
    #   - if a refit is due: refit GARCH-t per currency on data[:t] and store params
    #   - get sigma_{t+1|t} for each currency using params fit on data through t
    #   - rank, build inverse-vol leg weights
    #   - record weights; costs on turnover
    ...

# W_wf, pnl_wf, turnover_wf, costs_wf, sigma_wf = walk_forward(carry, rates_daily, d)

In [ ]:
# Checkpoint: compare pnl_wf to the in-sample vol-targeted pnl_vol (Section 4.2) over the same dates.
# A large gap in favor of the in-sample version is evidence of how much look-ahead was flattering it.

## 5.2 Performance Report: The Tearsheet

**VaR note:** to get a *basket-level* 95% conditional VaR, the simplest route is a GARCH-$t$ on the basket's own P&L, walk-forward like 5.1. `arch`'s Student-$t$ is standardized to unit variance, so rescale the `scipy` quantile by $\sqrt{(\nu-2)/\nu}$.

In [ ]:
# TODO: walk-forward one-step-ahead 95% VaR series for the basket
var95 = ...

# TODO: tearsheet dict / DataFrame with every metric in the table below
#   turnover annualized = total turnover / years;  costs as % of initial capital
tearsheet = ...

| Metric | Value |
|---|---|
| Annualized return | |
| Annualized volatility | |
| Sharpe ratio | |
| **Skewness** | |
| **Excess kurtosis** | |
| Maximum drawdown | |
| 95% conditional VaR (GARCH-$t$) | |
| Total turnover (annualized) | |
| Total costs + funding spread paid (% of initial capital) | |

*(TODO: fill in from the cell above, or render the DataFrame as markdown programmatically.)*

In [ ]:
# TODO: naive benchmark: long every currency equally vs USD, no ranking (W = 1/9 each)
# TODO: cumulative P&L plot: walk-forward basket vs AUD/JPY vs naive benchmark (labels, legend, title)
# TODO: drawdown plot underneath

In [ ]:
# TODO: VaR backtest: share of days where the loss exceeded the VaR forecast (target ~5%); plot P&L with -VaR band

### Crash-episode zoom-in
Pick an episode inside your sample. With `fetch_data.py`'s 2010 start, candidates are Jan 2015 (SNB floor removal, CHF), Aug 2015 (China devaluation, risk-off), Mar 2020 (COVID), and Aug 2024 (JPY carry unwind). 2008 Q3–Q4 is only available if you moved `START` earlier.
Plot over that window: basket P&L / drawdown and the GARCH VaR forecast. Did the risk estimate rise *before* the losses or only after?

In [ ]:
CRASH_START, CRASH_END = ..., ...   # TODO
# TODO: 2-panel zoom: cumulative P&L (or drawdown) | daily P&L with -VaR line

### PM Recommendation (TODO, 3–5 sentences)

## 5.3 Interpretation (TODO)

- **Vol targeting:** real reduction in drawdown / tail risk vs. equal-weight, or cosmetic? *(cite numbers)*
- **Frictions:** how much did costs and funding spread change the conclusion vs. Sections 4.1/4.2?
- **"Steady edge" or "nickels in front of a steamroller"?** Defend it with skewness, kurtosis, and the crash zoom.
- **First change you'd make running this book**, and why.

---
# 6. Bonus: Trend Overlay (up to +5)

Take the carry-implied position in currency $i$ only if trend agrees:
- long only if spot is above its trailing moving average (3m or 12m; pick and justify),
- short only if spot is below it,
- otherwise flat (or reduced size; justify).

The trend signal must also be computed with data through the rebalance date only.

In [ ]:
# TODO: trend signal at rebalance dates (dates x ccys), +1 / -1
# TODO: W_trend = W_wf masked where sign(W_wf) != trend signal; re-normalize legs (or not, and say why)
# TODO: tearsheet comparison with vs without overlay, especially skew, max DD, and the crash window

**Interpretation (TODO):** Did trend help, hurt, or mostly trade average return for a smoother path?

---
# Appendix: Reproducibility

- Data as-of date and package versions (cell below).
- Cache files are not committed; delete the cached CSVs in `data/` (or pass `force=True`) to force a fresh pull.

In [ ]:
import importlib
for pkg in ["numpy", "pandas", "statsmodels", "arch", "yfinance", "scipy"]:
    try:
        print(pkg, importlib.import_module(pkg).__version__)
    except Exception as e:
        print(pkg, "not found")
print("Run date:", date.today())